In [ ]:
import os
import json
import boto3
import pandas as pd
from pyspark.sql.functions import col, explode, from_json
from pyspark.sql.types import ArrayType, StringType

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = ""

In [ ]:
# df = spark.read.parquet("s3://curso-kyrsten-udemy-datalake/bronze/mongodb/accounts/")

# Exemplo de outras extrações
# df_pokemon = spark.read.option("multiline", "true").json("s3://curso-kyrsten-udemy-datalake/bronze/api/pokemon/")
# df_pg_supabase = spark.read.parquet("s3://curso-kyrsten-udemy-datalake/bronze/postgres/Pokemon")

In [ ]:
# Baixa o arquivo do S3 para o disco local do Databricks
s3 = boto3.client("s3")
bucket = "curso-kyrsten-udemy-datalake"
key = "bronze/mongodb/movies/movies.parquet"
local_path = "/tmp/movies.parquet"

s3.download_file(bucket, key, local_path)
print(f"Baixado: {local_path}")

In [ ]:
# Lê com pandas
df_pandas = pd.read_parquet(local_path)
df_pandas = df_pandas[["title", "genres", "cast"]]

In [ ]:
# Converte para Spark DataFrame
df = spark.createDataFrame(df_pandas)

In [ ]:
def str_to_list(x):
    if isinstance(x, str) and x.strip().startswith("["):
        try:
            return json.loads(x)
        except (json.JSONDecodeError, TypeError):
            return []
    if isinstance(x, list):
        return x
    return []

arrays_cols = ["genres", "cast"]

for column in arrays_cols:
    df_pandas[column] = df_pandas[column].apply(str_to_list)

In [ ]:
df = spark.createDataFrame(df_pandas)
df_exploded_genres = df.withColumn("genre", explode(col("genres")))
df_exploded_cast = df_exploded_genres.withColumn("cast_unique", explode(col("cast")))

In [ ]:
# Padronização do nome das colunas podem ser feitas nessa camada
df_final = df_exploded_cast \
    .withColumnRenamed("cast", "casts") \
    .withColumnRenamed("cast_unique", "cast") \
    .select("title", "genre", "cast") # Exclusão das colunas de arrays que não serão utilizadas